# Caso 08 — El Mirador, hotel: ¿lleno es lo mismo que rentable?
## Día 4 — El descubrimiento: RevPAR neto y GOPPAR

Hoy cerramos la escalera de métricas en SQL (DuckDB) subiendo los dos últimos peldaños que deciden la inversión de Elena Miralles (CEO):
1. **El patrón robusto de las dos CTEs (`por_hotel` y `costes`):**
   * `costes_operativos` tiene 12 filas por hotel (una por mes). Hacer un `JOIN` directo entre `reservas` y `costes_operativos` multiplicaría cada reserva por 12. Por eso agregamos primero las reservas y comisiones en una CTE (`por_hotel`) y los costes en otra (`costes`), uniéndolas al final por `hotel_id`.
2. **Peldaño 4 — RevPAR Neto:**
   $$\text{RevPAR Neto} = \frac{\text{Ingreso de habitación} - \text{Comisiones del canal}}{\text{Noches disponibles}}$$
3. **Peldaño 5 — GOPPAR (Beneficio Operativo Bruto por Habitación Disponible):**
   $$\text{GOPPAR} = \frac{\text{Ingreso} - \text{Comisiones} - (\text{Coste variable/noche} \times \text{Noches vendidas}) - \text{Coste fijo anual}}{\text{Noches disponibles}}$$
4. **El contraste que decide la inversión y el contrafactual:**
   * Comparamos la **cuota de capacidad** (% de noches disponibles del grupo) frente a la **cuota de beneficio operativo (GOP)**.
   * Calculamos cuánto dinero adicional ganaría El Mirador si **Costa** rindiese en GOPPAR al mismo nivel que **Playa Sur** (el otro resort de playa del grupo).


In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()
for tabla in ["hoteles", "tipos_habitacion", "canales", "costes_operativos", "clientes", "reservas"]:
    con.execute(f"CREATE OR REPLACE VIEW {tabla} AS SELECT * FROM read_csv_auto('data/clean/{tabla}.csv')")

# Escalera completa de métricas (Peldaños 1 al 5) con CTEs separadas
df_escalera = con.execute("""
WITH por_hotel AS (
    SELECT
        r.hotel_id,
        COUNT(*) AS reservas_consumidas,
        SUM(r.noches) AS noches_vendidas,
        SUM(r.noches * r.tarifa_noche) AS ingreso_bruto,
        SUM(r.noches * r.tarifa_noche * COALESCE(c.comision_pct, 0)) AS comision_total,
        SUM(CASE WHEN r.canal IN ('booking', 'expedia', 'turoperador') THEN r.noches ELSE 0 END) AS noches_ota
    FROM reservas r
    LEFT JOIN canales c ON r.canal = c.nombre
    WHERE r.estado IN ('confirmada', 'completada')
      AND r.fecha_cancelacion IS NULL
      AND r.tarifa_noche > 0 AND r.tarifa_noche < 2000
      AND r.noches > 0 AND r.num_habitaciones > 0
      AND (r.fecha_entrada IS NULL OR r.fecha_entrada <= DATE '2024-12-31')
      AND (r.fecha_salida IS NULL OR r.fecha_entrada IS NULL OR r.fecha_salida >= r.fecha_entrada)
    GROUP BY r.hotel_id
),
costes AS (
    SELECT
        hotel_id,
        SUM(coste_fijo) AS coste_fijo_anual,
        AVG(coste_variable_noche) AS coste_variable_noche
    FROM costes_operativos
    GROUP BY hotel_id
)
SELECT
    h.hotel_id,
    h.nombre,
    h.tipo,
    h.noches_disponibles,
    p.noches_vendidas,
    -- Peldaño 1: Ocupación
    ROUND(100.0 * p.noches_vendidas / h.noches_disponibles, 2) AS ocupacion_pct,
    -- Peldaño 2: ADR
    ROUND(p.ingreso_bruto / p.noches_vendidas, 2) AS adr_eur,
    -- Peldaño 3: RevPAR Bruto
    ROUND(p.ingreso_bruto / h.noches_disponibles, 2) AS revpar_bruto_eur,
    -- Comisión efectiva
    ROUND(100.0 * p.comision_total / p.ingreso_bruto, 2) AS comision_efectiva_pct,
    -- Peldaño 4: RevPAR Neto
    ROUND((p.ingreso_bruto - p.comision_total) / h.noches_disponibles, 2) AS revpar_neto_eur,
    -- Costes operativos (variable total + fijo anual)
    ROUND(co.coste_variable_noche * p.noches_vendidas, 2) AS coste_variable_total_eur,
    ROUND(co.coste_fijo_anual, 2) AS coste_fijo_anual_eur,
    -- Peldaño 5: GOPPAR y Beneficio Operativo Total (GOP)
    ROUND((p.ingreso_bruto - p.comision_total - (co.coste_variable_noche * p.noches_vendidas) - co.coste_fijo_anual) / h.noches_disponibles, 2) AS goppar_eur,
    ROUND(p.ingreso_bruto - p.comision_total - (co.coste_variable_noche * p.noches_vendidas) - co.coste_fijo_anual, 2) AS gop_total_eur
FROM hoteles h
JOIN por_hotel p ON h.hotel_id = p.hotel_id
JOIN costes co ON h.hotel_id = co.hotel_id
ORDER BY goppar_eur DESC
""").df()

display(df_escalera)


,hotel_id,nombre,tipo,noches_disponibles,noches_vendidas,ocupacion_pct,adr_eur,revpar_bruto_eur,comision_efectiva_pct,revpar_neto_eur,coste_variable_total_eur,coste_fijo_anual_eur,goppar_eur,gop_total_eur
0,3,El Mirador Centro,urbano,60000,40800.0,68.0,135.0,91.8,6.0,86.29,816000.0,1080000.0,54.69,3281520.0
1,2,El Mirador Playa Sur,vacacional,60000,42000.0,70.0,120.0,84.0,7.0,78.12,1050000.0,1080000.0,42.62,2557200.0
2,4,El Mirador Ciudad,urbano,55000,44000.0,80.0,100.0,80.0,13.0,69.60,880000.0,990000.0,35.60,1958000.0
3,1,El Mirador Costa,vacacional,80000,68000.0,85.0,90.0,76.5,15.0,65.03,1768000.0,1440000.0,24.93,1994000.0
4,5,El Mirador Sierra,urbano,40000,23200.0,58.0,85.0,49.3,10.0,44.37,510400.0,720000.0,13.61,544400.0


### Análisis de los Peldaños 4 (RevPAR Neto) y 5 (GOPPAR)
La escalera completa muestra cómo se rompe el espejismo de **El Mirador Costa** peldaño a peldaño:
1. **En el Peldaño 4 (`RevPAR Neto`):**
   * Al descontar la comisión del canal, **Centro** sube a **$86,29	ext{ €}$**, **Playa Sur** a **$78,12	ext{ €}$**, **Ciudad** a **$69,60	ext{ €}$** y **Costa** cae a **$65,03	ext{ €}$** (`65.025 €`).
   * La brecha entre Centro y Costa, que en RevPAR bruto era de $15,30	ext{ €}$, se ensancha hasta **$21,26	ext{ €}$ por habitación disponible** porque el 15 % de comisión de Costa le come $11,48	ext{ €/hab. disp.}$, frente a solo $5,51	ext{ €}$ en Centro.
2. **En el Peldaño 5 (`GOPPAR`, la métrica de la casa):**
   * Al descontar además el coste variable de cada noche ocupada (`26 €/noche` en Costa por `68.000` noches = $1.768.000	ext{ €}$) y el coste fijo anual ($1.440.000	ext{ €}$), **El Mirador Costa se hunde hasta $24,93	ext{ €}$ de GOPPAR (`24.925 €`)**, quedando **penúltimo del grupo** (solo por delante del pequeño hotel de 3 estrellas de Sierra, con $13,61	ext{ €}$).
   * **El Mirador Centro** lidera con **$54,69	ext{ €}$ de GOPPAR** —**más del doble ($2,19	imes$) que Costa** teniendo 17 puntos menos de ocupación— seguido de **Playa Sur ($42,62	ext{ €}$)** y **Ciudad ($35,60	ext{ €}$)**.


In [2]:
# El contraste que decide la inversión: Cuota de Capacidad vs Cuota de Beneficio (GOP) y Contrafactual
df_inversion = con.execute("""
WITH por_hotel AS (
    SELECT
        r.hotel_id,
        SUM(r.noches) AS noches_vendidas,
        SUM(r.noches * r.tarifa_noche) AS ingreso_bruto,
        SUM(r.noches * r.tarifa_noche * COALESCE(c.comision_pct, 0)) AS comision_total
    FROM reservas r
    LEFT JOIN canales c ON r.canal = c.nombre
    WHERE r.estado IN ('confirmada', 'completada')
      AND r.fecha_cancelacion IS NULL
      AND r.tarifa_noche > 0 AND r.tarifa_noche < 2000
      AND r.noches > 0 AND r.num_habitaciones > 0
      AND (r.fecha_entrada IS NULL OR r.fecha_entrada <= DATE '2024-12-31')
      AND (r.fecha_salida IS NULL OR r.fecha_entrada IS NULL OR r.fecha_salida >= r.fecha_entrada)
    GROUP BY r.hotel_id
),
costes AS (
    SELECT
        hotel_id,
        SUM(coste_fijo) AS coste_fijo_anual,
        AVG(coste_variable_noche) AS coste_variable_noche
    FROM costes_operativos
    GROUP BY hotel_id
),
resumen AS (
    SELECT
        h.hotel_id,
        h.nombre,
        h.noches_disponibles,
        p.noches_vendidas,
        p.ingreso_bruto,
        (p.ingreso_bruto - p.comision_total - (co.coste_variable_noche * p.noches_vendidas) - co.coste_fijo_anual) AS gop_total
    FROM hoteles h
    JOIN por_hotel p ON h.hotel_id = p.hotel_id
    JOIN costes co ON h.hotel_id = co.hotel_id
)
SELECT
    nombre,
    noches_disponibles,
    ROUND(100.0 * noches_disponibles / SUM(noches_disponibles) OVER (), 1) AS cuota_capacidad_pct,
    ROUND(ingreso_bruto, 2) AS ingreso_bruto_eur,
    ROUND(100.0 * ingreso_bruto / SUM(ingreso_bruto) OVER (), 1) AS cuota_ingreso_pct,
    ROUND(gop_total / noches_disponibles, 2) AS goppar_eur,
    ROUND(gop_total, 2) AS gop_total_eur,
    ROUND(100.0 * gop_total / SUM(gop_total) OVER (), 1) AS cuota_beneficio_pct,
    ROUND((100.0 * gop_total / SUM(gop_total) OVER ()) - (100.0 * noches_disponibles / SUM(noches_disponibles) OVER ()), 1) AS brecha_beneficio_vs_capacidad_pp
FROM resumen
ORDER BY gop_total_eur DESC
""").df()

display(df_inversion)

# Resumen global del grupo y cálculo contrafactual (¿Qué pasaría si Costa rindiera como Playa Sur?)
gop_grupo = df_inversion["gop_total_eur"].sum()
disp_grupo = df_inversion["noches_disponibles"].sum()
goppar_medio_grupo = gop_grupo / disp_grupo

goppar_playa_sur = df_inversion.loc[df_inversion["nombre"] == "El Mirador Playa Sur", "goppar_eur"].iloc[0]
gop_actual_costa = df_inversion.loc[df_inversion["nombre"] == "El Mirador Costa", "gop_total_eur"].iloc[0]
disp_costa = df_inversion.loc[df_inversion["nombre"] == "El Mirador Costa", "noches_disponibles"].iloc[0]
mejora_contrafactual = (goppar_playa_sur * disp_costa) - gop_actual_costa

print(f"Beneficio Operativo Total (GOP) del grupo : {gop_grupo:,.2f} €")
print(f"GOPPAR medio ponderado del grupo          : {goppar_medio_grupo:,.2f} €")
print(f"Contrafactual (si Costa tuviera el GOPPAR de Playa Sur, 42,62 €): +{mejora_contrafactual:,.2f} € adicionales al año")


,nombre,noches_disponibles,cuota_capacidad_pct,ingreso_bruto_eur,cuota_ingreso_pct,goppar_eur,gop_total_eur,cuota_beneficio_pct,brecha_beneficio_vs_capacidad_pp
0,El Mirador Centro,60000,20.3,5508000.0,23.9,54.69,3281520.0,31.8,11.4
1,El Mirador Playa Sur,60000,20.3,5040000.0,21.9,42.62,2557200.0,24.7,4.4
2,El Mirador Costa,80000,27.1,6120000.0,26.6,24.93,1994000.0,19.3,-7.8
3,El Mirador Ciudad,55000,18.6,4400000.0,19.1,35.60,1958000.0,18.9,0.3
4,El Mirador Sierra,40000,13.6,1972000.0,8.6,13.61,544400.0,5.3,-8.3


Beneficio Operativo Total (GOP) del grupo : 10,335,120.00 €
GOPPAR medio ponderado del grupo          : 35.03 €
Contrafactual (si Costa tuviera el GOPPAR de Playa Sur, 42,62 €): +1,415,600.00 € adicionales al año


### Conclusión del Día 4: El contraste que decide el presupuesto de Elena
1. **Asignación de recursos (`Cuota de Capacidad` vs `Cuota de Beneficio`):**
   * **El Mirador Costa** consume el **27,1 % de la capacidad del grupo** (`80.000` de `295.000` noches disponibles) pero aporta solo el **19,3 % del beneficio operativo** ($1.994.000	ext{ €}$ de $10.335.120	ext{ €}$ totales, una brecha de **$-7,8	ext{ pp}$**).
   * **El Mirador Centro** tiene el **20,3 % de la capacidad** (`60.000` noches disponibles) y aporta el **31,8 % del beneficio operativo** ($3.281.520	ext{ €}$, superando su peso en **$+11,5	ext{ pp}$**).
   * **El Mirador Playa Sur** también tiene el **20,3 % de la capacidad** y aporta el **24,7 % del beneficio** ($2.557.200	ext{ €}$, más de medio millón por encima de Costa pese a ser un 25 % más pequeño).
2. **El tamaño del dinero dejado sobre la mesa (Contrafactual):**
   * La media del grupo es de **$35,03	ext{ €}$ de GOPPAR** ($10.335.120	ext{ €} / 295.000	ext{ noches}$).
   * Si **El Mirador Costa** rindiera por habitación como **Playa Sur** ($42,62	ext{ €}$, que también es un resort vacacional de playa de 4 estrellas pero con venta directa y tarifa protegida), generaría **$3.409.600	ext{ €}$** en lugar de $1.994.000	ext{ €}$: **+$1.415.600	ext{ €}$ ($\sim 1,42	ext{ M€}$) más de beneficio operativo al año**.
